In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [ ]:
AMINO_ACID_CLASSES = {
    "polarity": {
        "hydrophobic": ["A", "V", "L", "I", "M", "F", "W", "P", "G"],
        "polar": ["S", "T", "Y", "C", "N", "Q"],
        "charged": ["K", "R", "H", "D", "E"],
    },

    "charge": {
        "positive": ["K", "R", "H"],
        "negative": ["D", "E"],
        "neutral": [
            "A", "V", "L", "I", "M", "F", "W", "P", "G",
            "S", "T", "Y", "C", "N", "Q"
        ],
    },

    "chemical_type": {
        "aliphatic": ["G", "A", "V", "L", "I"],
        "aromatic": ["F", "Y", "W"],
        "hydroxyl": ["S", "T", "Y"],
        "acidic": ["D", "E"],
        "amide": ["N", "Q"],
        "basic": ["K", "R", "H"],
        "sulfur": ["C", "M"],
        "imino": ["P"],
        # "other": [],  # placeholder (no gaps)
    },

    "essentiality": {
        "essential": ["F", "V", "T", "W", "M", "L", "I", "K", "H"],
        "non_essential": ["A", "N", "D", "E", "Q", "G", "P", "S", "C", "Y"],
    },

    "aromaticity": {
        "aromatic": ["F", "Y", "W"],
        "non_aromatic": [
            "A", "R", "N", "D", "C", "E", "Q", "G", "H", "I",
            "L", "K", "M", "P", "S", "T", "V"
        ],
    },

    "side_chain_size": {
        "small": ["G", "A", "S", "T", "P"],
        "medium": ["C", "N", "D", "Q", "E"],
        "large": ["V", "L", "I", "M", "F", "Y", "W", "K", "R", "H"],
    },
}

GRAM_POSITIVE = [
    "S. aureus ATCC 12600",
    "S. aureus (ATCC BAA-1556) - MRSA",
    "vancomycin-resistant E. faecalis ATCC 700802",
    "vancomycin-resistant E. faecium ATCC 700221",
    "L. monocytogenes ATCC 19111 (BEIRES NR-106)",
    "C. aerofaciens ATCC25986",
    "C. scindens ATCC35704",
    "C. spiroforme ATCC29900",
    "E. rectale ATCC33656",
    "C. symbiosum",
    "R. obeum",
    "R. torques",
]

GRAM_NEGATIVE = [
    "A. baumannii ATCC 19606",
    "E. coli ATCC 11775",
    "E. coli AIG221",
    "E. coli AIG222",
    "K. pneumoniae ATCC 13883",
    "P. aeruginosa PA01",
    "P. aeruginosa PA14",
    "E. coli Nissle",
    "Salmonella enterica ATCC 9150 (BEIRES NR-515)",
    "Salmonella enterica (BEIRES NR-170)",
    "Salmonella enterica ATCC 9150 (BEIRES NR-174)",

    "A. muciniphila ATCC BAA-835",
    "B. fragilis ATCC25285",
    "B. vulgatus ATCC8482",
    "B. thetaiotaomicron ATCC29148",
    "B. thetaiotaomicron Complemmented",
    "B. thetaiotaomicron Mutant",
    "B. uniformis ATCC8492",
    "B. eggerthi ATCC27754",
    "B. ovatus ATCC8483",
    "P. distasonis ATCC8503",
    "P. copri DSMZ18205",
]

In [ ]:
ALL_AA = sorted("ACDEFGHIKLMNPQRSTVWY")

In [ ]:
import numpy as np
import pandas as pd
from typing import Callable


def aggregate_mutation_dict(
    data_dict: dict,
    value_key: str | None = None,
    subset: list[str] | None = None,      # REQUIRED
    agg_func: Callable | None = np.mean,
    agg_func_kwargs: dict | None = None
):
    """
    Aggregates mutation dictionaries of the structure:
        { (from_aa, to_aa): { value_key: { variable_name: list_of_values } } }

    Parameters
    ----------
    data_dict : dict
        Full mutation dictionary.
    value_key : str, optional
        The inner key to aggregate over (e.g. "diff"). If None, assumes data_dict
        has structure { (from_aa, to_aa): { variable_name: list_of_values } }.
    subset : list of str, optional
        List of variable names to include (e.g. species names). If None, includes all.
    agg_func : callable, default np.mean
        Aggregation function to apply over the subset variables (e.g., np.mean, np.max, 
        np.min, lambda x: np.quantile(x, 0.95), etc.).
    agg_func_kwargs : dict, optional
        Additional keyword arguments to pass to the aggregation function.

    Returns
    -------
    df_mean, df_std, df_n : pivot tables
        Pivot tables with aggregated values (using agg_func), std, and n.
    """

    rows = []

    for (aa1, aa2), record in data_dict.items():
        # Handle optional value_key
        if value_key is not None:
            if value_key not in record:
                continue
            value_dict = record[value_key]
        else:
            value_dict = record

        all_values = []

        # include only selected variables (or all if subset is None)
        for var, vals in value_dict.items():
            if subset is None or var in subset:
                all_values.append(vals)

        if len(all_values) == 0:
            mean = np.nan
            std = np.nan
            n = 0
        else:
            arr = np.array(all_values, dtype=float)
            agg_arr = agg_func(arr, **agg_func_kwargs if agg_func_kwargs else {}) if agg_func else arr
            assert agg_arr.ndim == 1, f"Aggregation function {agg_func} returned array with {agg_arr.ndim} dimensions"
            mean = float(np.mean(agg_arr))
            std = float(np.std(agg_arr))
            n = len(agg_arr)

        rows.append({
            "from_aa": aa1,
            "to_aa": aa2,
            "mean": mean,
            "std": std,
            "n": n,
        })

    df = pd.DataFrame(rows)

    df_mean = df.pivot(index="from_aa", columns="to_aa", values="mean")
    df_std  = df.pivot(index="from_aa", columns="to_aa", values="std")
    df_n    = df.pivot(index="from_aa", columns="to_aa", values="n")

    return df_mean, df_std, df_n

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
def aa_mutation_heatmap(
    df_mean: pd.DataFrame,
    aa_list: list[str],
    df_std: pd.DataFrame | None = None,
    df_n: pd.DataFrame | None = None,
    AA_group: dict[str, list[str]] | None = None,
    cmap: str = "magma",
    nan_color: str = "black",
    figsize=(9, 7),
    fontsize: int = 11,
    title_fontsize: int = 12,
    label_fontsize: int = 11,
    title: str = "",
    x_label: str = "To AA n-gram",
    y_label: str = "From AA n-gram",
    group_linewidth: float = 1.5,
    group_colors: dict[str, str] | None = None,

    # NEW: separate axis control
    x_bar_gap: float = 0.0,       # heatmap → X-axis color bar
    x_label_gap: float = 0.0,     # color bar → X-axis label
    y_bar_gap: float = 0.0,       # heatmap → Y-axis color bar
    y_label_gap: float = 0.0,     # color bar → Y-axis label
    group_label_color: str = "black",
    
    # NEW: colorbar control
    vmin: float | None = None,    # minimum value for colorbar
    vmax: float | None = None,    # maximum value for colorbar
    cbar_shrink: float = 0.8,     # shrink factor for colorbar (0-1)
    cbar_aspect: float = 20,      # aspect ratio of colorbar
    cbar_pad: float = 0.02,       # padding between heatmap and colorbar
    
    # NEW: threshold highlighting
    df_threshold: pd.DataFrame | None = None,  # dataframe with values to compare against threshold
    threshold: float | None = None,            # threshold value
    threshold_direction: str = "both",         # "above", "below", or "both"
    threshold_color: str = "red",              # color of threshold rectangle
    threshold_linewidth: float = 2.0,          # linewidth of threshold rectangle
):
    """
    Heatmap with fully symmetric and independently controlled
    X/Y axis group bars and group labels.
    """

    df = df_mean.copy()

    # === 1. Group sorting ===
    if AA_group is not None:
        sorted_aa = []
        group_sizes, group_names = [], []

        for gname, members in AA_group.items():
            present = [aa for aa in aa_list if aa in members]
            if present:
                sorted_aa.extend(present)
                group_sizes.append(len(present))
                group_names.append(gname)

        df = df.loc[sorted_aa, sorted_aa]
        if df_std is not None: df_std = df_std.loc[sorted_aa, sorted_aa]
        if df_n is not None:   df_n   = df_n.loc[sorted_aa, sorted_aa]
        if df_threshold is not None: df_threshold = df_threshold.loc[sorted_aa, sorted_aa]
    else:
        sorted_aa = aa_list
        group_sizes, group_names = [], []
        if df_threshold is not None:
            df_threshold = df_threshold.loc[sorted_aa, sorted_aa]


    # === 2. Annotation matrix ===
    annot = np.empty(df.shape, dtype=object)
    for i, r in enumerate(df.index):
        for j, c in enumerate(df.columns):
            if pd.isna(df.loc[r, c]):
                annot[i, j] = ""
                continue
            txt = f"{df.loc[r, c]:.2f}"
            if df_std is not None:
                sd = df_std.loc[r, c]
                if not pd.isna(sd):
                    txt += f"\n± {sd:.2f}"
            if df_n is not None:
                n = df_n.loc[r, c]
                if not pd.isna(n):
                    txt += f"\n(n={int(n)})"
            annot[i, j] = txt

    # === 3. NaN color ===
    base_cmap = plt.get_cmap(cmap).copy()
    base_cmap.set_bad(color=nan_color)
    mask = pd.isna(df)

    # === 4. Draw heatmap ===
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        df,
        cmap=base_cmap,
        annot=annot,
        fmt="",
        mask=mask,
        square=True,
        cbar=True,
        vmin=vmin,
        vmax=vmax,
        cbar_kws={
            "shrink": cbar_shrink,
            "aspect": cbar_aspect,
            "pad": cbar_pad,
        },
        annot_kws={"fontsize": fontsize - 1},
        ax=ax,
    )

    ax.set_title(title, fontsize=title_fontsize)
    ax.set_xlabel(x_label, fontsize=label_fontsize)
    ax.set_ylabel(y_label, fontsize=label_fontsize)
    ax.tick_params(axis="both", labelsize=label_fontsize)

    # === 4.5. Threshold highlighting ===
    if df_threshold is not None and threshold is not None:
        # Determine which cells to highlight
        highlight_mask = pd.DataFrame(False, index=df_threshold.index, columns=df_threshold.columns)
        
        if threshold_direction == "above":
            highlight_mask = df_threshold > threshold
        elif threshold_direction == "below":
            highlight_mask = df_threshold < threshold
        elif threshold_direction == "both":
            highlight_mask = (df_threshold > threshold) | (df_threshold < threshold)
        else:
            raise ValueError(f"threshold_direction must be 'above', 'below', or 'both', got '{threshold_direction}'")
        
        # Draw rectangles around highlighted cells
        # Note: heatmap cells are centered at integer positions (0.5, 1.5, 2.5, ...)
        for i, row_idx in enumerate(df_threshold.index):
            for j, col_idx in enumerate(df_threshold.columns):
                if highlight_mask.loc[row_idx, col_idx] and not pd.isna(df_threshold.loc[row_idx, col_idx]):
                    # Draw rectangle around the cell
                    # Cell boundaries: [i, i+1] x [j, j+1] in data coordinates
                    rect = plt.Rectangle(
                        (j, i),           # bottom-left corner
                        1,                 # width
                        1,                 # height
                        fill=False,
                        edgecolor=threshold_color,
                        linewidth=threshold_linewidth,
                        transform=ax.transData,
                    )
                    ax.add_patch(rect)

    # === 5. Group boundaries + bars + labels (AXES COORDINATES) ===
    if AA_group is not None and group_sizes:

        boundaries = np.cumsum(group_sizes)
        mids = boundaries - np.array(group_sizes) / 2
        total = len(sorted_aa)

        # Convert mid positions from data coords → axes coords
        mids_axes = mids / total
        size_axes = np.array(group_sizes) / total

        bar_thickness_axes = 0.02  # thickness as fraction of axis

        # Separator lines (still in data coordinates)
        for b in boundaries[:-1]:
            ax.axhline(b, color="black", lw=group_linewidth)
            ax.axvline(b, color="black", lw=group_linewidth)

        # Draw bars + labels in AXES COORDINATES
        for mid_ax, gname, gsize_ax in zip(mids_axes, group_names, size_axes):

            col = group_colors.get(gname, "lightgray") if group_colors else "lightgray"

            # === Y-axis color bar ===
            # Reverse Y-axis position to match heatmap order (top to bottom in data = bottom to top in axes)
            mid_ax_y_reversed = 1.0 - mid_ax
            ax.add_patch(plt.Rectangle(
                ( -y_bar_gap, mid_ax_y_reversed - gsize_ax/2 ),
                bar_thickness_axes,
                gsize_ax,
                transform=ax.transAxes,
                clip_on=False,
                color=col,
            ))

            # === Y-axis label ===
            ax.text(
                -y_bar_gap - bar_thickness_axes - y_label_gap,
                mid_ax_y_reversed,
                gname,
                ha="center",
                va="center",
                transform=ax.transAxes,
                rotation=90,
                fontsize=fontsize,
                color=group_label_color,
            )

            # === X-axis color bar (BOTTOM) ===
            ax.add_patch(plt.Rectangle(
                (mid_ax - gsize_ax/2, -x_bar_gap),
                gsize_ax,
                bar_thickness_axes,
                transform=ax.transAxes,
                clip_on=False,
                color=col,
            ))

            # === X-axis label (BOTTOM) ===
            ax.text(
                mid_ax,
                -x_bar_gap - bar_thickness_axes - x_label_gap,
                gname,
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=fontsize,
                color=group_label_color,
            )


    fig.tight_layout()
    return fig, ax

In [ ]:
from scipy.stats import mannwhitneyu, wilcoxon
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
from typing import Callable


def mutation_mannwhitney_test(
    dict_a: dict,
    dict_b: dict | None = None,  # NEW: optional, if None test against zero
    value_key: str = None,
    subset: list[str] = None,                        # REQUIRED
    alternative: str = "two-sided",
    agg_func: Callable = np.mean,                    # NEW: aggregation function
    agg_func_kwargs: dict | None = None,              # NEW: kwargs for aggregation function

    # NEW: kwargs
    mannwhitney_kwargs: dict | None = None,   # passed into mannwhitneyu
    wilcoxon_kwargs: dict | None = None,      # passed into wilcoxon (when dict_b is None)
    multitest: bool = False,                  # whether to apply BH correction
    multitest_kwargs: dict | None = None,     # kwargs passed into multipletests

):
    """
    Performs Mann–Whitney U test for each mutation (aa1 -> aa2)
    comparing values from dict_a vs dict_b.
    
    If dict_b is None, performs one-sample Wilcoxon signed-rank test
    against zero for values from dict_a.

    Supports optional multiple testing correction (Benjamini–Hochberg).

    Parameters
    ----------
    dict_a : dict
        Mutation dictionary.
    dict_b : dict or None
        Mutation dictionary. If None, tests dict_a values against zero.
    value_key : str
        Inner key to extract values from.
    subset : list[str]
        REQUIRED list of variable names to include.
    alternative : str
        two-sided, less, greater (Mann–Whitney argument)
    agg_func : callable, default np.mean
        Aggregation function to apply over the subset variables (e.g., np.mean, np.max, 
        np.min, lambda x: np.quantile(x, 0.95), etc.).
    agg_func_kwargs : dict, optional
        Additional keyword arguments to pass to the aggregation function.
    mannwhitney_kwargs : dict
        Additional arguments forwarded to scipy.stats.mannwhitneyu.
    wilcoxon_kwargs : dict
        Additional arguments forwarded to scipy.stats.wilcoxon (when dict_b is None).
    multitest : bool
        Whether to apply multiple testing correction (Benjamini–Hochberg).
    multitest_kwargs : dict
        Extra keyword arguments passed to statsmodels.multipletests().

    Returns
    -------
    df_p_raw : pivot table of raw p-values
    df_p_adj : pivot table of corrected p-values (NaN if multitest=False)
    df_u      : pivot table of U statistics (or W statistics if one-sample)
    df_n1     : pivot table of sample counts in dict_a
    df_n2     : pivot table of sample counts in dict_b (NaN if one-sample)
    df_effect : pivot table of median difference (median(a) - median(b)) or median(a) if one-sample
    """

    if mannwhitney_kwargs is None:
        mannwhitney_kwargs = {}
    if wilcoxon_kwargs is None:
        wilcoxon_kwargs = {}
    if multitest_kwargs is None:
        multitest_kwargs = {}
    if agg_func_kwargs is None:
        agg_func_kwargs = {}

    rows = []
    one_sample = dict_b is None

    # union of all mutation pairs
    if one_sample:
        all_keys = set(dict_a.keys())
    else:
        all_keys = set(dict_a.keys()).union(dict_b.keys())

    for (aa1, aa2) in all_keys:

        def extract_values(rec):
            if rec is None or value_key not in rec:
                return []
            all_values = []
            for var, vals in rec[value_key].items():
                if var in subset:
                    all_values.append(vals)
            
            if len(all_values) == 0:
                return []
            
            arr = np.array(all_values, dtype=float)
            agg_arr = agg_func(arr, **agg_func_kwargs)
            assert agg_arr.ndim == 1, f"Aggregation function {agg_func} returned array with {agg_arr.ndim} dimensions"
            return list(agg_arr)
        
        vals_a = extract_values(dict_a.get((aa1, aa2)))
        
        if one_sample:
            # One-sample test against zero
            if len(vals_a) == 0:
                w_stat = np.nan
                p_val = np.nan
                effect = np.nan
            else:
                try:
                    w_stat, p_val = wilcoxon(
                        vals_a,
                        alternative=alternative,
                        **wilcoxon_kwargs
                    )
                    effect = np.median(vals_a)  # median value (difference from zero)
                except Exception:
                    w_stat = np.nan
                    p_val = np.nan
                    effect = np.nan

            rows.append({
                "from_aa": aa1,
                "to_aa": aa2,
                "p_raw": p_val,
                "u": w_stat,  # storing W statistic in 'u' column for consistency
                "n1": len(vals_a),
                "n2": np.nan,
                "effect": effect,
            })
        else:
            # Two-sample test
            vals_b = extract_values(dict_b.get((aa1, aa2)))

            # if one side has zero samples → cannot test
            if len(vals_a) == 0 or len(vals_b) == 0:
                u_stat = np.nan
                p_val = np.nan
                effect = np.nan
            else:
                try:
                    u_stat, p_val = mannwhitneyu(
                        vals_a,
                        vals_b,
                        alternative=alternative,
                        **mannwhitney_kwargs
                    )
                    effect = np.median(vals_a) - np.median(vals_b)

                except Exception:
                    u_stat = np.nan
                    p_val = np.nan
                    effect = np.nan

            rows.append({
                "from_aa": aa1,
                "to_aa": aa2,
                "p_raw": p_val,
                "u": u_stat,
                "n1": len(vals_a),
                "n2": len(vals_b),
                "effect": effect,
            })

    df = pd.DataFrame(rows)

    # --- Multiple testing correction ---
    if multitest:
        pvals = df["p_raw"].values
        _, p_adj, _, _ = multipletests(
            pvals,
            method="fdr_bh",       # Benjamini-Hochberg
            **multitest_kwargs
        )
        df["p_adj"] = p_adj
    else:
        df["p_adj"] = np.nan

    # --- Make pivot tables ---
    df_p_raw = df.pivot(index="from_aa", columns="to_aa", values="p_raw")
    df_p_adj = df.pivot(index="from_aa", columns="to_aa", values="p_adj")
    df_u     = df.pivot(index="from_aa", columns="to_aa", values="u")
    df_n1    = df.pivot(index="from_aa", columns="to_aa", values="n1")
    df_n2    = df.pivot(index="from_aa", columns="to_aa", values="n2")
    df_effect = df.pivot(index="from_aa", columns="to_aa", values="effect")

    return df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect

---

In [ ]:
diff_values_path = Path('/home/pszmk/pep-compass/results/mutation_analysis/hydramp_x_ampsphere/direction_threshold=0.001_token_threshold=0.1_jacobian_mode=approx_jacobian_eps=0.05/ngram1/diff_values.csv')
diff_df = pd.read_csv(diff_values_path)

In [ ]:
from scripts.mutation_analysis.analysis import compute_mutation_statistics_from_df, compute_ranks_for_single_sample
from scripts.mutation_analysis.mutations import compute_aggregate_mutation_counter

---

In [ ]:
group_colors = {
    "hydrophobic": "#F28E8E",
    "polar": "#8EC9F2",
    "charged": "#A9E68E",
}

#### 1-grams

In [ ]:
aggregate_counter_ngram1 = compute_aggregate_mutation_counter(diff_df=diff_df,
parent_col='parent',
mutant_col='mutant',
ngram_size=1,
allow_ngrams_overlap=True)

mutation_statistics_ngram1 = compute_mutation_statistics_from_df(diff_df=diff_df,
aggregate_counter=aggregate_counter_ngram1,
parent_col='parent',
mutant_col='mutant',
value_cols=[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
ngram_size=1,
allow_ngrams_overlap=True)

In [ ]:
def aa_mutation_heatmap_factory_for_some_preset(subset, title, mutation_statistics ):
    transition_mean, transition_std, transition_n = aggregate_mutation_dict(mutation_statistics, value_key='diff', subset=subset, agg_func=np.mean, agg_func_kwargs={'axis': 0})

    df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect = mutation_mannwhitney_test(
        dict_a=mutation_statistics,
        value_key='diff',
        subset=subset,
        agg_func=np.mean,  # matches aggregate_mutation_dict
        agg_func_kwargs={'axis': 0},
        multitest=True,
    )

    return aa_mutation_heatmap(
        df_mean=transition_mean,
        aa_list=ALL_AA,
        df_std=transition_std,
        df_n=transition_n,
        df_threshold=df_p_adj,
        threshold=0.001,
        threshold_direction="below",
        threshold_color="blue",
        AA_group=AMINO_ACID_CLASSES["polarity"],
        cmap="RdYlGn_r",
        nan_color="white",
        figsize=(24, 24),
        fontsize=11,
        title=title,
        x_label="To AA n-gram",
        y_label="From AA n-gram",
        group_linewidth=4,
        group_colors=group_colors,
        x_bar_gap = 0.07,
        y_bar_gap = 0.07,
        title_fontsize=30,
        label_fontsize=18,
    )

In [ ]:
aa_mutation_heatmap_factory_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    title="All strains | log2 diff | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

In [ ]:
aa_mutation_heatmap_factory_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE],
    title="Gram + | log2 diff | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

In [ ]:
aa_mutation_heatmap_factory_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_NEGATIVE],
    title="Gram - | log2 diff | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

### Ranks

In [ ]:
ranks_ngram1 = compute_ranks_for_single_sample(
    stats_dict=mutation_statistics_ngram1,
    aggregate_cols=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    aggregate_counter=aggregate_counter_ngram1,
    aggregation_func="mean")
gp_ranks_ngram1 = compute_ranks_for_single_sample(
    stats_dict=mutation_statistics_ngram1,
    aggregate_cols=[f"{b}_log2" for b in GRAM_POSITIVE],
    aggregate_counter=aggregate_counter_ngram1,
    aggregation_func="mean"
)
gn_ranks_ngram1 = compute_ranks_for_single_sample(
    stats_dict=mutation_statistics_ngram1,
    aggregate_cols=[f"{b}_log2" for b in GRAM_NEGATIVE],
    aggregate_counter=aggregate_counter_ngram1,
    aggregation_func="mean"
)

In [ ]:
def aa_mutation_heatmap_factory_for_ranks_relative_for_some_preset(subset, title, mutation_statistics, reference_mutation_statistics):
    transition_mean, transition_std, transition_n = aggregate_mutation_dict(mutation_statistics, value_key='diff', subset=subset, agg_func=np.mean, agg_func_kwargs={'axis': 0})

    ranks = compute_ranks_for_single_sample(
        stats_dict=mutation_statistics,
        aggregate_cols=subset,
        aggregate_counter=aggregate_counter_ngram1,
        aggregation_func="mean"
        )
    reference_ranks = compute_ranks_for_single_sample(
        stats_dict=reference_mutation_statistics,
        aggregate_cols=subset,
        aggregate_counter=aggregate_counter_ngram1,
        aggregation_func="mean"
        )
    
    # Subtract reference ranks from ranks
    comp_ranks = {}
    for (from_aa, to_aa), rank_value in ranks.items():
        reference_value = reference_ranks.get((from_aa, to_aa), 0)
        print(reference_value)
        comp_ranks[(from_aa, to_aa)] = rank_value - reference_value

    # transition_ranks, _, _ = aggregate_mutation_dict(ranks, value_key=None, subset=[f"{a}_rank" for a in subset], agg_func=None)
    # reference_transition_ranks, _, _ = aggregate_mutation_dict(reference_ranks, value_key=None, subset=[f"{a}_rank" for a in subset], agg_func=None)

    return aa_mutation_heatmap(
        df_mean=comp_ranks,
        aa_list=ALL_AA,
        df_std=None,
        df_n=transition_n,
        threshold_direction="below",
        threshold_color="blue",
        AA_group=AMINO_ACID_CLASSES["polarity"],
        cmap="RdYlGn_r",
        nan_color="white",
        figsize=(24, 24),
        fontsize=11,
        title=title,
        x_label="To AA n-gram",
        y_label="From AA n-gram",
        group_linewidth=4,
        group_colors=group_colors,
        x_bar_gap = 0.07,
        y_bar_gap = 0.07,
        title_fontsize=30,
        label_fontsize=18,
    )

In [ ]:
aa_mutation_heatmap_factory_for_ranks_relative_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    title="All strains | ranks | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
    reference_mutation_statistics=mutation_statistics_ngram1,
)


In [ ]:
ranks = compute_ranks_for_single_sample(
        stats_dict=mutation_statistics_ngram1,
        aggregate_cols=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
        aggregate_counter=aggregate_counter_ngram1,
        aggregation_func="mean"
        )

In [ ]:
transition_ranks, _, _ = aggregate_mutation_dict(ranks, value_key=None, subset=[f"{b}_log2_rank" for b in GRAM_POSITIVE + GRAM_NEGATIVE], agg_func=None)

In [ ]:
aa_mutation_heatmap(
        df_mean=transition_ranks,
        aa_list=ALL_AA,
        df_std=None,
        df_n=None,
        threshold_direction="below",
        threshold_color="blue",
        AA_group=AMINO_ACID_CLASSES["polarity"],
        cmap="RdYlGn_r",
        nan_color="white",
        figsize=(24, 24),
        fontsize=11,
        # title=title,
        x_label="To AA n-gram",
        y_label="From AA n-gram",
        group_linewidth=4,
        group_colors=group_colors,
        x_bar_gap = 0.07,
        y_bar_gap = 0.07,
        title_fontsize=30,
        label_fontsize=18,
    )

In [ ]:
# transition_mean, transition_std, transition_n = aggregate_mutation_dict(mutation_statistics, value_key='diff', subset=subset, agg_func=np.mean, agg_func_kwargs={'axis': 0})

#     ranks = compute_ranks_for_single_sample(
#         stats_dict=mutation_statistics,
#         aggregate_cols=subset,
#         aggregate_counter=aggregate_counter_ngram1,
#         aggregation_func="mean"
#         )

#     transition_ranks, _, _ = aggregate_mutation_dict(ranks, value_key=None, subset=subset, agg_func=np.mean, agg_func_kwargs={'axis': 0})

#     return aa_mutation_heatmap(
#         df_mean=transition_ranks,
#         aa_list=ALL_AA,
#         df_std=None,
#         df_n=None,
#         threshold_direction="below",
#         threshold_color="blue",
#         AA_group=AMINO_ACID_CLASSES["polarity"],
#         cmap="RdYlGn_r",
#         nan_color="white",
#         figsize=(24, 24),
#         fontsize=11,
#         title=title,
#         x_label="To AA n-gram",
#         y_label="From AA n-gram",
#         group_linewidth=4,
#         group_colors=group_colors,
#         x_bar_gap = 0.07,
#         y_bar_gap = 0.07,
#         title_fontsize=30,
#         label_fontsize=18,
#     )

In [ ]:
transition_ranks

In [ ]:
aa_mutation_heatmap_factory_for_ranks_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    title="All strains | ranks | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

In [ ]:
mutation_statistics_ngram1

In [ ]:
aa_mutation_heatmap_factory_for_some_preset(
    subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
    title="All strains | log2 diff | 1-gram",
    mutation_statistics=mutation_statistics_ngram1,
)

### Check out 3-grams

In [ ]:
aggregate_counter_ngram3 = compute_aggregate_mutation_counter(diff_df=diff_df,
parent_col='parent',
mutant_col='mutant',
ngram_size=3,
allow_ngrams_overlap=True)

mutation_statistics_ngram3 = compute_mutation_statistics_from_df(diff_df=diff_df,
aggregate_counter=aggregate_counter_ngram3,
parent_col='parent',
mutant_col='mutant',
value_cols=[f"{b}_log2_diff" for b in GRAM_POSITIVE + GRAM_NEGATIVE],
ngram_size=3,
allow_ngrams_overlap=True)

### Subpopulation comparison

In [ ]:
def aa_mutation_heatmap_factory_relative_for_some_preset(subset, title, mutation_statistics, mutation_statistics_reference):
    transition_mean, transition_std, transition_n = aggregate_mutation_dict(mutation_statistics, value_key='diff', subset=subset, agg_func=np.mean, agg_func_kwargs={'axis': 0})

    df_p_raw, df_p_adj, df_u, df_n1, df_n2, df_effect = mutation_mannwhitney_test(
        dict_a=mutation_statistics,
        dict_b=mutation_statistics_reference,
        value_key='diff',
        subset=subset,
        agg_func=np.mean,  # matches aggregate_mutation_dict
        agg_func_kwargs={'axis': 0},
        multitest=True,
    )

    return aa_mutation_heatmap(
        df_mean=transition_mean,
        aa_list=ALL_AA,
        df_std=transition_std,
        df_n=transition_n,
        df_threshold=df_p_adj,
        threshold=0.001,
        threshold_direction="below",
        threshold_color="blue",
        AA_group=AMINO_ACID_CLASSES["polarity"],
        cmap="RdYlGn_r",
        nan_color="white",
        figsize=(24, 24),
        fontsize=11,
        title=title,
        x_label="To AA n-gram",
        y_label="From AA n-gram",
        group_linewidth=4,
        group_colors=group_colors,
        x_bar_gap = 0.07,
        y_bar_gap = 0.07,
        title_fontsize=30,
        label_fontsize=18,
    )

### Check out different aggregation functions over subset of variables

In [ ]:
gp_transition_mean_agg_median, gp_transition_std_agg_median, gp_transition_n_agg_median = aggregate_mutation_dict(mutation_statistics_ngram1, value_key='diff', subset=[f"{b}_log2" for b in GRAM_POSITIVE], agg_func=np.median)
gn_transition_mean_agg_median, gn_transition_std_agg_median, gn_transition_n_agg_median = aggregate_mutation_dict(mutation_statistics_ngram1, value_key='diff', subset=[f"{b}_log2" for b in GRAM_NEGATIVE], agg_func=np.median)
transition_mean_agg_median, transition_std_agg_median, transition_n_agg_median = aggregate_mutation_dict(mutation_statistics_ngram1, value_key='diff', subset=[f"{b}_log2" for b in GRAM_POSITIVE + GRAM_NEGATIVE], agg_func=np.median)

---

In [ ]:
from collections import defaultdict

# Marginalize over the second variable (sum all counts for each unique second element)
marginal_second = defaultdict(int)
for (first, second), count in transition_counts.items():
    marginal_second[second] += count

# Convert to regular dict and sort by value (descending)
marginal_second = dict(sorted(marginal_second.items(), key=lambda x: x[1], reverse=True))

marginal_second

---

In [ ]:
df = pd.DataFrame([
    {'from_aa': k[0], 'to_aa': k[1], 'count': v}
    for k, v in transition_counts.items()
])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Build dataframe
df = pd.DataFrame([
    {'from_aa': k[0], 'to_aa': k[1], 'count': v}
    for k, v in transition_counts.items()
])

# Add a label string
df["label"] = df["from_aa"] + "→" + df["to_aa"]

# Sort
df_sorted = df.sort_values("count", ascending=False)

# Convert label to *ordered* categorical
df_sorted["label"] = pd.Categorical(
    df_sorted["label"],
    categories=df_sorted["label"],   # this keeps original order!
    ordered=True
)

plt.figure(figsize=(8, 48))
sns.barplot(
    data=df_sorted,
    x="count",
    y="label",     # now seaborn respects the order
    orient="h"
)
plt.xlabel("Count")
plt.ylabel("Mutation")
plt.title("Sorted mutation counts")
plt.tight_layout()
plt.show()
